In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import logging
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(override=True)

sys.path.append(str(Path().cwd().parent))

from claude_agent_sdk import ClaudeAgentOptions

from src.agent import run_agent
from src.bq import BigQuery
from src.gcs import CloudStorage

In [ ]:
WS = Path().cwd().parent / "ws"
ROLES = json.loads((Path().cwd().parent / "roles.json").read_text())

In [ ]:
def build_options(role: str) -> ClaudeAgentOptions:
    config = ROLES[role]
    servers = {}
    if "bq" in config["servers"]:
        servers["bq"] = BigQuery().mcp()
    if "gcs" in config["servers"]:
        servers["gcs"] = CloudStorage().mcp()
    return ClaudeAgentOptions(
        cwd=str(WS),
        setting_sources=["project"],
        mcp_servers=servers,
        thinking={"type": "adaptive", "display": "summarized"},
        permission_mode=config["permission_mode"],
        max_turns=config["max_turns"],
    )


async def main(prompt, role="ops") -> None:
    logging.basicConfig(
        level=os.environ.get("LOG_LEVEL", "INFO"),
        format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    )
    logging.getLogger("mcp").setLevel(logging.WARNING)
    logging.getLogger("httpx").setLevel(logging.WARNING)

    await run_agent(prompt, build_options(role), echo=True)

In [5]:
await main("what's the weather like in tokyo?")

2026-07-28 15:33:58,885 INFO claude_agent_sdk._internal.transport.subprocess_cli: Using bundled Claude Code CLI: /Users/yo/Developer/repo/naxos/.venv/lib/python3.13/site-packages/claude_agent_sdk/_bundled/claude
⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


[thinking] The user is asking about Tokyo's weather, which is outside the scope of this repository. I need current information, so I'll use the web search tool to find that data.
[tool] ToolSearch: {'query': 'select:WebSearch', 'max_results': 1}
[thinking] I'll search for Tokyo's current weather conditions.
[tool] WebSearch: {'query': 'Tokyo weather today forecast'}


2026-07-28 15:34:15,103 INFO agent: run complete: cost_usd=0.075516 turns=3 error=False


Tokyo today (July 28) — hot and humid, as expected for late July:

- **Now:** ~29°C, overcast, humidity ~69%
- **Today:** mostly cloudy, high around 31°C (88°F), spotty afternoon showers
- **Tonight:** a thunderstorm possible in the evening, low around 24°C (76°F)

Worth having an umbrella on hand.

Sources:
- [Weather.com — Tokyo](https://weather.com/weather/today/l/a2d49e31992a3679724161c3a9029211ecefeda3e1da24489637a5685ded3fdf)
- [NearWeather — Tokyo](https://www.nearweather.com/location/tokyo-jp-1850147)
